# Sumas, promedios y el teorema central del límite

Cada distribución del capítulo anterior describía **una observación**
a la vez: una espera, una respuesta, un conteo. Pero ningún día de la
clínica se decide con un solo paciente, ni una encuesta se publica con
una sola respuesta, ni un turno de la línea se resume en una sola
pieza inspeccionada. Lo que termina importando es el **promedio** o
el **total** de muchas observaciones — y apenas damos ese paso, dos
preguntas insisten una y otra vez: ¿a qué tiende el promedio cuando
la muestra crece?, ¿cómo se distribuye alrededor de ese límite?

## Importaciones

In [ ]:
import math

import altair as alt
import pandas as pd

from core import (
    BinomialParams,
    ExponentialParams,
    NormalParams,
    Settings,
)
from distributions import (
    make_binomial,
    make_exponential,
    make_normal,
    tail_probability_of_continuous,
)
from distributions.evaluations import TailProbabilityInput
from exercises import NumericAnswerInput, verify_numeric_answer
from sampling import (
    CLTSimulationInput,
    GaltonBoardInput,
    LLNMultipleTrajectoriesInput,
    LLNSimulationInput,
    simulate_clt,
    simulate_galton_board,
    simulate_lln,
    simulate_lln_multiple_trajectories,
)
from visualization import (
    CLTComparisonChartInput,
    LLNChartInput,
    LLNMultipleTrajectoriesChartInput,
    chart_clt_comparison,
    chart_lln_multiple_trajectories,
    chart_lln_running_mean,
)
from visualization.theme import apply_theme
from widgets import (
    CLTExplorerInput,
    LLNExplorerInput,
    build_clt_explorer,
    build_lln_explorer,
)

In [ ]:
settings = Settings()

(sec-sums-lln)=
## La proporción observada se estabiliza

Volvemos al esquema Bernoulli ([](#eq-binomial-pmf) con $n = 1$).
Cada respuesta de la encuesta es $X_i \in \{0, 1\}$ con probabilidad
$p$ de ser «sí». El promedio acumulado tras $n$ respuestas es:

$$ \bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i $$ (eq-running-mean)

Esta es la misma definición que [](#eq-mean), pero ahora con $X_i$
**variables aleatorias**, no observaciones fijas.

**LLN (ley de los grandes números, versión débil):** si las $X_i$ son
i.i.d. con $E[X_i] = \mu$ y varianza finita,

$$ \bar{X}_n \xrightarrow{P} \mu \quad\text{cuando } n \to \infty $$ (eq-lln)


In [ ]:
bernoulli = make_binomial(BinomialParams(trials=1, success_probability=0.55))
lln_input = LLNSimulationInput(
    distribution=bernoulli,
    horizon=4_000,
    settings=settings,
)
lln_result = simulate_lln(lln_input)

lln_chart_input = LLNChartInput(
    lln_result=lln_result,
    title="Encuesta — proporción observada de «sí» (LLN)",
    settings=settings,
)
chart_lln_running_mean(lln_chart_input)

Una sola simulación podría haber bajado en lugar de subir, o haber pasado
más tiempo lejos de $0{,}55$. Para ver que la convergencia no depende de la
suerte de un único experimento, repetimos el mismo procedimiento muchas
veces en paralelo: cada trayectoria es una encuesta distinta hecha en una
ciudad imaginaria diferente, con sus propios «sí» y «no» en otro orden.


In [ ]:
lln_multi_input = LLNMultipleTrajectoriesInput(
    distribution=bernoulli,
    horizon=4_000,
    trajectory_count=30,
    settings=settings,
)
lln_multi_result = simulate_lln_multiple_trajectories(lln_multi_input)

lln_multi_chart_input = LLNMultipleTrajectoriesChartInput(
    lln_result=lln_multi_result,
    title="30 encuestas paralelas — todas convergen al mismo valor",
    settings=settings,
)
chart_lln_multiple_trajectories(lln_multi_chart_input)

El abanico es ancho al principio: con pocas respuestas, la proporción
observada en cada encuesta puede caer casi en cualquier lado. A medida que
$n$ crece, las treinta líneas se aprietan contra la media teórica $0{,}55$ y
el ancho de la franja se va a cero. Esa contracción simultánea es,
exactamente, lo que dice la ecuación [](#eq-lln): no importa con qué realización
concreta arranques, el promedio termina pegado al valor verdadero.


In [ ]:
lln_explorer_input = LLNExplorerInput(settings=settings)
build_lln_explorer(lln_explorer_input)

(sec-sums-galton)=
## El tablero de Galton

El **tablero de Galton** es la metáfora pictórica del TCL. Cada bola toma
$r$ decisiones independientes izquierda/derecha. La posición final es una
**suma** de $r$ pasos de $\pm 1$, y se distribuye aproximadamente Normal —
aunque cada paso individual sea uniforme. Es un ejemplo deliberadamente alejado
de tiempos, encuestas o turnos: justamente por eso ayuda a aislar el TCL del
fenómeno particular que se esté midiendo.


In [ ]:
galton_input = GaltonBoardInput(rows=24, balls=8_000, settings=settings)
galton_result = simulate_galton_board(galton_input)

galton_table = pd.DataFrame({
    "posicion": galton_result.bin_positions,
    "frecuencia": galton_result.bin_counts,
})
galton_chart = (
    alt
    .Chart(galton_table)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("posicion:O", title="Casilla final"),
        y=alt.Y("frecuencia:Q", title="Cantidad de bolas"),
    )
    .properties(title="Tablero de Galton — suma de 24 pasos ±1")
)
apply_theme(galton_chart, settings)

(sec-sums-clt)=
## El TCL formal

Sean $X_1, X_2, \dots$ i.i.d. con $E[X_i] = \mu$ y
$\mathrm{Var}(X_i) = \sigma^2 < \infty$. **Paso 1:** centramos el promedio
[](#eq-running-mean) restándole su esperanza $\mu$. **Paso 2:** lo escalamos por el desvío
estándar de $\bar{X}_n$, que es $\sigma/\sqrt{n}$. **Paso 3:** afirmamos:

$$ \begin{aligned}
Z_n &= \frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \\[4pt]
Z_n &\xrightarrow{d} \mathcal{N}(0, 1)
\end{aligned} $$ (eq-clt)

Notar que [](#eq-clt) **no** dice que $X_i$ tienda a una Normal. Dice que la
**media estandarizada** lo hace.


(sec-sums-clt-waits)=
## TCL aplicado a tiempos de espera

Cada espera es Exponencial ([](#eq-exp-pdf)), claramente **no**
Normal: tiene cola larga y es asimétrica. Pero el promedio de 30 esperas se
vuelve aproximadamente Normal por [](#eq-clt). Lo materializamos simulando muchas
réplicas:


In [ ]:
clinic_distribution = make_exponential(ExponentialParams(rate=0.25))
clt_input = CLTSimulationInput(
    distribution=clinic_distribution,
    sample_size_per_replicate=30,
    replicates=5_000,
    settings=settings,
)
clinic_clt_result = simulate_clt(clt_input)

clt_chart_input = CLTComparisonChartInput(
    clt_result=clinic_clt_result,
    title="Promedio de 30 esperas (clínica) — convergencia a la Normal",
    settings=settings,
)
chart_clt_comparison(clt_chart_input)

Las medias estandarizadas de 5.000 días simulados se alinean con la
$\mathcal{N}(0, 1)$ ([](#eq-normal-pdf)). El TCL [](#eq-clt) **predice**
esta forma, no la impone artificialmente.


In [ ]:
clt_explorer_input = CLTExplorerInput(settings=settings)
build_clt_explorer(clt_explorer_input)

(sec-sums-binomial-normal)=
## Aproximación Binomial → Normal

Si $Y \sim \text{Bin}(n, p)$, podemos escribir $Y$ como suma de $n$
Bernoullis i.i.d. Aplicando [](#eq-clt) (con $\mu = p$ y $\sigma^2 = p(1-p)$)
y multiplicando por $n$:

$$ \begin{aligned}
Y       &\approx \mathcal{N}\!\bigl(np,\ np(1-p)\bigr) \\[4pt]
        &\quad\text{válida si } np \ge 10 \text{ y } n(1-p) \ge 10
\end{aligned} $$ (eq-clt-bin)

Para una jornada de inspección con $n = 100$ y $p = 0{,}4$:


In [ ]:
factory_normal_approx = make_normal(NormalParams(mean=40.0, standard_deviation=math.sqrt(24.0)))
factory_tail_input = TailProbabilityInput(
    distribution=factory_normal_approx,
    upper_bound=45.0,
)
factory_tail_probability = tail_probability_of_continuous(factory_tail_input)
factory_tail_probability

## Ejercicio 1 — Desvío del promedio

Si $X_i \sim \mathcal{N}(50, 100)$ (es decir $\sigma = 10$) y tomamos
$n = 25$ observaciones, ¿cuál es el desvío estándar de $\bar{X}_n$?

**Idea.** Aplicamos el factor de escala que aparece en [](#eq-clt):

$$ \sigma_{\bar{X}_n} = \frac{\sigma}{\sqrt{n}} = \frac{10}{\sqrt{25}} = 2 $$


In [ ]:
expected_standard_error = 10.0 / math.sqrt(25)

student_answer_se = 2.0
verify_input = NumericAnswerInput(
    student_answer=student_answer_se,
    expected_answer=expected_standard_error,
)
verify_numeric_answer(verify_input)

## Ejercicio 2 — Aproximar Binomial con Normal

$Y \sim \text{Bin}(100, 0{,}4)$. Aproximá $P(Y \le 45)$ aplicando [](#eq-clt-bin)
(sin corrección por continuidad para simplificar):

$$ \begin{aligned}
E[Y]              &= np \\[4pt]
                  &= 40 \\[4pt]
\mathrm{Var}(Y)   &= np(1-p) \\[4pt]
                  &= 24 \\[4pt]
P(Y \le 45)       &\approx P\!\left(Z \le \frac{45 - 40}{\sqrt{24}}\right) \\[4pt]
                  &\approx P(Z \le 1{,}02) \approx 0{,}846
\end{aligned} $$


In [ ]:
exercise_normal_approx = make_normal(NormalParams(mean=40.0, standard_deviation=math.sqrt(24.0)))
exercise_tail_input = TailProbabilityInput(
    distribution=exercise_normal_approx,
    upper_bound=45.0,
)
expected_probability = tail_probability_of_continuous(exercise_tail_input).probability

student_answer_probability = 0.846
verify_input = NumericAnswerInput(
    student_answer=student_answer_probability,
    expected_answer=expected_probability,
)
verify_numeric_answer(verify_input)

Hasta acá el flujo siempre fue el mismo: dados los parámetros — una tasa, una
proporción real, un desvío estándar — calcular probabilidades sobre las
observaciones. En la práctica, sin embargo, los parámetros casi nunca se
conocen; lo que se tiene son los datos, y la pregunta natural es la
**inversa**: dado lo medido, ¿qué se puede afirmar de lo que no vemos? ¿Cuál
es el rango plausible para la verdadera espera media de la clínica?, ¿cuánto
se acerca la proporción observada en la encuesta a la verdadera?, ¿alcanzan
los conteos de la línea de producción para descartar la afirmación de que los
defectos están bajo el 5%? Ese cambio de sentido — de los parámetros a los
datos a los parámetros otra vez — es el oficio de la **inferencia**.
